# Exploratory Data Analysis & Model Accuracy Evaluation
This notebook downloads the Autism Detection dataset (age 3-5) from Roboflow, cleans and maps class labels based on filename prefixes, balances the dataset using oversampling, trains a YOLOv8 detection model (configured for Google Drive backup and recovery), runs inference, evaluates accuracy metrics using a robust Background class mapping, and demonstrates custom memory hibernation techniques.

In [ ]:
# 1. Install required packages
!pip install roboflow requests pandas matplotlib opencv-python numpy seaborn scikit-learn ultralytics

In [ ]:
# 2. Import libraries
import os
import cv2
import glob
import json
import base64
import shutil
import random
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from roboflow import Roboflow
from collections import Counter
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
# 3. Download the dataset from Roboflow
API_KEY = "DFcDFKgDOZ2j6bjYYSNg"
MODEL_ID = "autism-detection-age-3-5-3gjk8/2"

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("autismdetection-wr5ri").project("autism-detection-age-3-5-3gjk8")
version = project.version(2)
dataset = version.download("yolov8")

# Dataset Cleaning (Remapping Classes from Filename Prefixes)
Since YOLO exports default bounding boxes to Class 0, this cell inspects the filename prefixes to correctly map labels: images starting with 'Non_Autism' are mapped to Class 1 (Non-Autism), and images starting with 'Autism' are mapped to Class 0 (Autism).

In [ ]:
dataset_dir = dataset.location
yaml_path = os.path.join(dataset_dir, "data.yaml")

# Read data.yaml to verify class names
with open(yaml_path, "r") as f:
    yaml_content = f.read()
print("Original data.yaml:")
print(yaml_content)

splits = ["train", "valid", "test"]

for split in splits:
    lbl_dir = os.path.join(dataset_dir, split, "labels")
    img_dir = os.path.join(dataset_dir, split, "images")
    lbl_files = glob.glob(os.path.join(lbl_dir, "*.txt"))

    for lbl_file in lbl_files:
        base_name = os.path.splitext(os.path.basename(lbl_file))[0]
        
        # Determine target class ID based on filename prefix
        if base_name.lower().startswith("non"):
            target_class_id = 1  # Non-Autism
        else:
            target_class_id = 0  # Autism
            
        with open(lbl_file, "r") as f:
            lines = f.readlines()

        new_lines = []
        for line in lines:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            # Overwrite class ID with filename prefix class mapping
            new_lines.append(f"{target_class_id} {' '.join(parts[1:])}\n")

        with open(lbl_file, "w") as f:
            f.writelines(new_lines)

    print(f"{split}: remapped label files from filename prefixes.")

# Write updated data.yaml
new_yaml_content = f"""train: {os.path.join(dataset_dir, 'train', 'images')}
val: {os.path.join(dataset_dir, 'valid', 'images')}
test: {os.path.join(dataset_dir, 'test', 'images')}

nc: 2
names: ['Autism', 'Non-Autism']
"""

with open(yaml_path, "w") as f:
    f.write(new_yaml_content)

print("\nUpdated data.yaml:")
print(new_yaml_content)

class_names = ["Autism", "Non-Autism"]

# Dataset Balancing (Oversampling & Undersampling)
To prevent model bias/mode collapse (e.g. model guessing only the majority class), we can automatically oversample or undersample the dataset. Oversampling duplicates the minority class images/labels with unique names, preserving valuable training data.

In [ ]:
# Set to True to balance training folder files
BALANCE_DATASET = True
BALANCE_STRATEGY = "oversample"  # "oversample" keeps all data; "undersample" deletes majority files

dataset_dir = dataset.location

if BALANCE_DATASET:
    train_img_dir = os.path.join(dataset_dir, "train", "images")
    train_lbl_dir = os.path.join(dataset_dir, "train", "labels")

    # Group training files by EVERY class present (not just first line)
    class_to_files = {}
    lbl_files = glob.glob(os.path.join(train_lbl_dir, "*.txt"))

    for lbl_file in lbl_files:
        base_name = os.path.splitext(os.path.basename(lbl_file))[0]
        img_matches = glob.glob(os.path.join(train_img_dir, base_name + ".*"))
        if not img_matches:
            continue
        img_file = img_matches[0]

        with open(lbl_file, "r") as f:
            lines = [l for l in f.readlines() if l.strip()]
        if not lines:
            continue

        classes_in_file = set(int(line.strip().split()[0]) for line in lines)
        for cls_id in classes_in_file:
            class_to_files.setdefault(cls_id, []).append((lbl_file, img_file))

    if class_to_files:
        counts = {cls: len(files) for cls, files in class_to_files.items()}
        print("Class counts before balancing:", counts)
        majority_cls = max(counts, key=counts.get)
        target_count = counts[majority_cls]

        if BALANCE_STRATEGY == "undersample":
            minority_cls = min(counts, key=counts.get)
            target_count = counts[minority_cls]
            for cls, files in class_to_files.items():
                if len(files) > target_count:
                    to_remove = random.sample(files, len(files) - target_count)
                    for lbl, img in to_remove:
                        if os.path.exists(lbl): os.remove(lbl)
                        if os.path.exists(img): os.remove(img)
                    print(f"Removed {len(to_remove)} pairs from class {cls}")
                else:
                    print(f"Class {cls} kept as-is ({len(files)} files)")

        else:  # oversample — duplicate minority images/labels with unique filenames
            for cls, files in class_to_files.items():
                deficit = target_count - len(files)
                if deficit <= 0:
                    print(f"Class {cls} already at/above target ({len(files)})")
                    continue
                copies_needed = deficit
                i = 0
                added = 0
                while added < copies_needed:
                    lbl, img = files[i % len(files)]
                    ext = os.path.splitext(img)[1]
                    new_base = f"{os.path.splitext(os.path.basename(lbl))[0]}_dup{i}"
                    new_lbl = os.path.join(train_lbl_dir, new_base + ".txt")
                    new_img = os.path.join(train_img_dir, new_base + ext)
                    if not os.path.exists(new_lbl):
                        shutil.copy(lbl, new_lbl)
                        shutil.copy(img, new_img)
                        added += 1
                    i += 1
                print(f"Class {cls}: added {added} duplicated pairs (target {target_count})")
                
        print("Dataset balancing complete!")

In [ ]:
# 5. Parse dataset labels (verifies remaining 2 classes)
print("Dataset directories ready.")
print("Final class names:", class_names)

In [ ]:
# 6. Collect metadata from training, validation, and test splits
splits = ["train", "valid", "test"]
records = []

for split in splits:
    img_dir = os.path.join(dataset_dir, split, "images")
    lbl_dir = os.path.join(dataset_dir, split, "labels")
    
    img_files = glob.glob(os.path.join(img_dir, "*"))
    for img_file in img_files:
        base_name = os.path.splitext(os.path.basename(img_file))[0]
        lbl_file = os.path.join(lbl_dir, base_name + ".txt")
        
        # Get image sizes
        img = cv2.imread(img_file)
        if img is None:
            continue
        h, w, c = img.shape
        
        # Read label
        num_boxes = 0
        classes_in_image = []
        if os.path.exists(lbl_file):
            with open(lbl_file, "r") as f:
                lines = f.readlines()
            for line in lines:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls_id = int(parts[0])
                    classes_in_image.append(cls_id)
                    num_boxes += 1
                    
        records.append({
            "split": split,
            "image_path": img_file,
            "width": w,
            "height": h,
            "num_boxes": num_boxes,
            "classes": classes_in_image
        })

df = pd.DataFrame(records)
print(f"Parsed metadata for {len(df)} images.")

In [ ]:
# 7. Plot Split Distribution
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="split", palette="viridis")
plt.title("Image Count per Dataset Split")
plt.xlabel("Split")
plt.ylabel("Number of Images")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# 8. Class Distribution Analysis
all_classes = [c for classes in df["classes"] for c in classes]
class_counts = Counter(all_classes)

class_df = pd.DataFrame([
    {"Class Name": class_names[cid] if cid < len(class_names) else f"Class {cid}", "Count": count}
    for cid, count in class_counts.items()
])

plt.figure(figsize=(8, 5))
sns.barplot(data=class_df, x="Class Name", y="Count", palette="Set2")
plt.title("Distribution of Target Biomarker Classes")
plt.xlabel("Biomarker Class")
plt.ylabel("Annotation Bounding Box Count")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

print(class_df)

In [ ]:
# 9. Analyze Image Dimensions & Aspect Ratios
df["aspect_ratio"] = df["width"] / df["height"]
print("Image dimension summary stats:")
print(df[["width", "height", "aspect_ratio"]].describe())

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
sns.histplot(df["width"], bins=20, kde=True, color="blue")
plt.title("Image Width Distribution")
plt.subplot(1, 2, 2)
sns.histplot(df["height"], bins=20, kde=True, color="green")
plt.title("Image Height Distribution")
plt.tight_layout()
plt.show()

In [ ]:
# 10. Bounding Box Dimensions Analysis
box_records = []
for split in splits:
    img_dir = os.path.join(dataset_dir, split, "images")
    lbl_dir = os.path.join(dataset_dir, split, "labels")
    
    lbl_files = glob.glob(os.path.join(lbl_dir, "*.txt"))
    for lbl_file in lbl_files:
        with open(lbl_file, "r") as f:
            lines = f.readlines()
        for line in lines:
            parts = line.strip().split()
            if len(parts) >= 5:
                # YOLO format: cls_id, x_center, y_center, width, height (normalized)
                box_records.append({
                    "class": int(parts[0]),
                    "box_width": float(parts[3]),
                    "box_height": float(parts[4]),
                    "box_area": float(parts[3]) * float(parts[4])
                })

box_df = pd.DataFrame(box_records)
print(f"Collected metadata for {len(box_df)} bounding boxes.")

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.histplot(box_df["box_area"], bins=30, kde=True, color="purple")
plt.title("Bounding Box Area Distribution (Relative)")
plt.xlabel("Normalized Box Area")

plt.subplot(1, 2, 2)
sns.scatterplot(data=box_df, x="box_width", y="box_height", hue="class", palette="Set1", alpha=0.6)
plt.title("Bounding Box Width vs Height")
plt.xlabel("Normalized Width")
plt.ylabel("Normalized Height")
plt.tight_layout()
plt.show()

In [ ]:
# 11. Visualize Sample Images with Bounding Boxes
def draw_bounding_boxes(img_path, lbl_path, classes):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w, _ = img.shape
    
    if os.path.exists(lbl_path):
        with open(lbl_path, "r") as f:
            lines = f.readlines()
        for line in lines:
            parts = line.strip().split()
            if len(parts) >= 5:
                cls_id = int(parts[0])
                x_c, y_c, wb, hb = [float(x) for x in parts[1:5]]
                
                # Convert normalized coords back to absolute
                x1 = int((x_c - wb/2) * w)
                y1 = int((y_c - hb/2) * h)
                x2 = int((x_c + wb/2) * w)
                y2 = int((y_c + hb/2) * h)
                
                # Draw rectangle and text label
                label = classes[cls_id] if cls_id < len(classes) else f"Class {cls_id}"
                color = (0, 255, 0) if cls_id == 0 else (255, 0, 0)
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                cv2.putText(img, label, (x1, max(y1 - 10, 15)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    return img

# Pick random images and display them
sample_df = df[df["num_boxes"] > 0].sample(min(3, len(df)))

plt.figure(figsize=(15, 5))
for idx, (_, row) in enumerate(sample_df.iterrows()):
    base_name = os.path.splitext(os.path.basename(row["image_path"]))[0]
    lbl_file = os.path.join(dataset_dir, row["split"], "labels", base_name + ".txt")
    
    visualized = draw_bounding_boxes(row["image_path"], lbl_file, class_names)
    
    plt.subplot(1, 3, idx + 1)
    plt.imshow(visualized)
    plt.title(f"Split: {row['split']}")
    plt.axis("off")
plt.tight_layout()
plt.show()

# Colab Training Protection (Google Drive Mount)
This cell mounts your Google Drive so that YOLOv8 training outputs can be directly backed up. If your Google Colab instance disconnects, you can easily resume training without losing your progress.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    save_dir = "/content/drive/MyDrive/autism_yolo_runs"
except Exception as e:
    # Fallback to local directory when Drive mounting fails or is bypassed
    print(f"Google Drive mounting failed or bypassed. Fallback to local directory. Error: {e}")
    save_dir = "runs"

import os
os.makedirs(save_dir, exist_ok=True)
print(f"Runs will be stored at: {save_dir}")

# GPU Verification
This cell checks if Google Colab has successfully allocated a T4 GPU run-time instance, prints PyTorch details, and configures the active training device.

In [ ]:
import torch
from ultralytics import YOLO

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected. Training will be extremely slow (hours per epoch).")
    print("Go to Runtime > Change runtime type > GPU, then restart and re-run this cell.")

train_device = 0 if torch.cuda.is_available() else "cpu"

# Check if nvidia-smi works
!nvidia-smi

# Dataset Balance Check
This cell counts label instances per class in training and validation sets to verify dataset splits prior to model fine-tuning.

In [ ]:
def check_class_balance(labels_dir, class_names):
    """Counts label instances per class across all .txt YOLO label files."""
    counts = Counter()
    if not os.path.exists(labels_dir):
        print(f"Directory {labels_dir} does not exist. Skipping balance check.")
        return counts
    for fname in os.listdir(labels_dir):
        if not fname.endswith(".txt"):
            continue
        with open(os.path.join(labels_dir, fname)) as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                class_id = int(parts[0])
                counts[class_id] += 1
    total = sum(counts.values())
    print(f"\nLabel distribution in {labels_dir}:")
    for cid, name in enumerate(class_names):
        n = counts.get(cid, 0)
        pct = (n / total * 100) if total else 0
        print(f"  Class {cid} ({name}): {n} instances ({pct:.1f}%)")
    if total > 0:
        ratio = max(counts.values()) / max(min(counts.values(), default=1), 1)
        if ratio > 1.5:
            print(f"  ⚠️  Class imbalance detected (ratio {ratio:.2f}:1). "
                  f"Consider oversampling the minority class, adding more data for it, "
                  f"or using class-weighted loss.")
        else:
            print("  ✅ Classes are reasonably balanced.")
    return counts

train_labels_dir = os.path.join(dataset_dir, "train", "labels")
val_labels_dir = os.path.join(dataset_dir, "valid", "labels")

check_class_balance(train_labels_dir, class_names)
check_class_balance(val_labels_dir, class_names)

# Model Training (YOLOv8)
Below, we train a fresh YOLOv8 object detection model on the newly balanced 2-class dataset.

### Resume Training Support:
If your run disconnects midway, you can resume training by running:
```python
model = YOLO(f"{save_dir}/autism_run_v3/weights/last.pt")
results = model.train(resume=True)
```

In [ ]:
import torch
from ultralytics import YOLO

# Load a fresh YOLOv8 model
model = YOLO("yolov8n.pt")

# Train on your newly balanced dataset
results = model.train(
    data=yaml_path,
    epochs=150,
    imgsz=640,
    batch=16,
    optimizer="SGD",
    lr0=0.01,
    patience=30,
    project=save_dir,
    name="autism_run_v3",
    save=True,
    save_period=5,
    device=train_device,
    # data augmentation - helps significantly with small datasets
    mosaic=1.0,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    fliplr=0.5,
    degrees=10.0,
    translate=0.1,
    scale=0.5
)

# Evaluate on Validation Set (Used during training)
We reload the best trained checkpoint weights to evaluate metrics on the validation set split.

In [ ]:
best_path = os.path.join(save_dir, "autism_run_v3", "weights", "best.pt")
model = YOLO(best_path)

val_metrics = model.val(data=yaml_path, split="val")
print("\n--- Validation Set Results ---")
print("mAP50-95 per class:", val_metrics.box.maps)
print("Overall mAP50:", val_metrics.box.map50)
print("Overall mAP50-95:", val_metrics.box.map)
print("Confusion Matrix:\n", val_metrics.confusion_matrix.matrix)

# Evaluate on a Separate Held-Out Test Set (Generalization Check)
Here we run validation on the separate 'test' split. This checks whether validation repeats have introduced validation set overfitting.

In [ ]:
test_metrics = model.val(data=yaml_path, split="test")
print("\n--- Held-Out Test Set Results (true generalization) ---")
print("mAP50-95 per class:", test_metrics.box.maps)
print("Overall mAP50:", test_metrics.box.map50)
print("Confusion Matrix:\n", test_metrics.confusion_matrix.matrix)

# Bounding Box Custom Accuracy Evaluation (Background-Aware Matrix)
Below, we run local predictions on the test set and evaluate custom classification reports incorporating missed bounding boxes as 'No Detection / Background'.

In [ ]:
# 11. Define IoU Helper Function
def calculate_iou(box1, box2):
    x_left = max(box1[0], box2[0])
    y_top = max(box1[1], box2[1])
    x_right = min(box1[2], box2[2])
    y_bottom = min(box1[3], box2[3])

    if x_right < x_left or y_bottom < y_top:
        return 0.0

    intersection_area = (x_right - x_left) * (y_bottom - y_top)

    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])

    union_area = float(box1_area + box2_area - intersection_area)
    if union_area == 0.0:
        return 0.0
        
    return intersection_area / union_area

In [ ]:
# 12. Run Inference & Validate Predictions (Supports Offline Local Inference)
from ultralytics import YOLO

USE_LOCAL_MODEL = True
LOCAL_MODEL_PATH = os.path.join(save_dir, "autism_run_v3", "weights", "best.pt")
test_image_dir = os.path.join(dataset_dir, "test", "images")
test_label_dir = os.path.join(dataset_dir, "test", "labels")
test_images = glob.glob(os.path.join(test_image_dir, "*"))

local_model = None
if USE_LOCAL_MODEL:
    if os.path.exists(LOCAL_MODEL_PATH):
        print(f"Loading local model from: {LOCAL_MODEL_PATH}")
        local_model = YOLO(LOCAL_MODEL_PATH)
    else:
        print(f"Local weights not found at '{LOCAL_MODEL_PATH}'. Falling back to Roboflow API.")
        USE_LOCAL_MODEL = False

print(f"Found {len(test_images)} test images. Starting evaluation...")

y_true = []
y_pred = []
confidence_threshold = 0.30
iou_threshold = 0.45
BACKGROUND_CLASS = len(class_names)  # extra "no detection / background" bucket, NOT a real class

for img_path in test_images:
    base_name = os.path.splitext(os.path.basename(img_path))[0]
    lbl_path = os.path.join(test_label_dir, base_name + ".txt")

    gt_boxes = []
    if os.path.exists(lbl_path):
        with open(lbl_path, "r") as f:
            lines = f.readlines()
        for line in lines:
            parts = line.strip().split()
            if len(parts) >= 5:
                gt_boxes.append({
                    "class": int(parts[0]),
                    "x_center": float(parts[1]),
                    "y_center": float(parts[2]),
                    "width": float(parts[3]),
                    "height": float(parts[4])
                })

    pred_boxes = []
    if USE_LOCAL_MODEL and local_model is not None:
        results = local_model(img_path, verbose=False)
        for r in results:
            for box in r.boxes:
                conf = float(box.conf[0])
                if conf >= confidence_threshold:
                    x_c, y_c, pw, ph = box.xywhn[0].tolist()
                    pred_boxes.append({
                        "class": int(box.cls[0]),
                        "x_center": x_c, "y_center": y_c,
                        "width": pw, "height": ph,
                        "confidence": conf
                    })
    else:
        img = cv2.imread(img_path)
        if img is None:
            continue
        h, w, _ = img.shape
        _, img_encoded = cv2.imencode(".jpg", img)
        payload = base64.b64encode(img_encoded.tobytes()).decode("utf-8")
        roboflow_url = f"https://detect.roboflow.com/{MODEL_ID}?api_key={API_KEY}"
        try:
            resp = requests.post(roboflow_url, data=payload,
                                  headers={r"Content-Type": "application/x-www-form-urlencoded"})
            predictions = resp.json().get("predictions", []) if resp.status_code == 200 else []
        except Exception:
            predictions = []

        for pred in predictions:
            confidence = float(pred.get("confidence", 0))
            if confidence >= confidence_threshold:
                px_c, py_c = float(pred["x"]) / w, float(pred["y"]) / h
                pw, ph = float(pred["width"]) / w, float(pred["height"]) / h
                label = pred.get("class", "")
                # Map string label to numeric ID
                class_id = class_names.index(label) if label in class_names else 0
                pred_boxes.append({
                    "class": class_id, "x_center": px_c, "y_center": py_c,
                    "width": pw, "height": ph, "confidence": confidence
                })

    # Match predictions against ground truths using the background bucket format
    matched_gt = set()
    for p_box in pred_boxes:
        p_coords = [
            p_box["x_center"] - p_box["width"] / 2,
            p_box["y_center"] - p_box["height"] / 2,
            p_box["x_center"] + p_box["width"] / 2,
            p_box["y_center"] + p_box["height"] / 2
        ]
        best_iou, best_gt_idx = -1.0, -1
        for g_idx, g_box in enumerate(gt_boxes):
            if g_idx in matched_gt:
                continue
            g_coords = [
                g_box["x_center"] - g_box["width"] / 2,
                g_box["y_center"] - g_box["height"] / 2,
                g_box["x_center"] + g_box["width"] / 2,
                g_box["y_center"] + g_box["height"] / 2
            ]
            iou = calculate_iou(p_coords, g_coords)
            if iou > best_iou:
                best_iou, best_gt_idx = iou, g_idx

        if best_iou >= iou_threshold and best_gt_idx != -1:
            matched_gt.add(best_gt_idx)
            y_true.append(gt_boxes[best_gt_idx]["class"])
            y_pred.append(p_box["class"])
        else:
            y_true.append(BACKGROUND_CLASS)
            y_pred.append(p_box["class"])

    for g_idx, g_box in enumerate(gt_boxes):
        if g_idx not in matched_gt:
            y_true.append(g_box["class"])
            y_pred.append(BACKGROUND_CLASS)

print("Evaluation complete.")

In [ ]:
# 13. Generate Classification Report
target_names = class_names + ["No Detection / Background"]
unique_labels = sorted(list(set(y_true) | set(y_pred)))
valid_target_names = [target_names[i] for i in unique_labels]
report = classification_report(y_true, y_pred, target_names=valid_target_names, labels=unique_labels)
print("\n--- CLASSIFICATION ACCURACY REPORT ---")
print(report)

In [ ]:
# 14. Plot Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(7, 5.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=valid_target_names, yticklabels=valid_target_names)
plt.title("Confusion Matrix - Autism Screening Bounding Boxes")
plt.ylabel("Actual Classification (Ground Truth)")
plt.xlabel("Predicted Classification (YOLOv8 Model)")
plt.show()

In [ ]:
# 15. Clean up downloaded dataset files to save space
if os.path.exists(dataset_dir):
    print("Cleaning up downloaded dataset directory:", dataset_dir)
    shutil.rmtree(dataset_dir)
    print("Cleanup complete!")

# Model Hibernation Demonstration (Memory Optimization)
Below is the exact code that runs on your backend server to achieve model hibernation. It lazy-loads the weights file on request, and unloads it to clear RAM/VRAM after 5 minutes of inactivity.

In [ ]:
import gc
import time
import torch
from ultralytics import YOLO

class HibernatingModel:
    def __init__(self, model_path, idle_timeout=300):
        self.model_path = model_path
        self.idle_timeout = idle_timeout
        self.model = None
        self.last_used = 0.0

    def get_model(self):
        self.last_used = time.time()
        if self.model is None:
            print(f"Loading custom model '{self.model_path}' into memory...")
            self.model = YOLO(self.model_path)
        return self.model

    def check_inactivity(self):
        if self.model is not None and (time.time() - self.last_used) >= self.idle_timeout:
            print(f"Model idle timeout reached. Hibernating '{self.model_path}' to free RAM/VRAM...")
            self.model = None
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            return True
        return False

# Create local instanced demo
detector_demo = HibernatingModel(LOCAL_MODEL_PATH, idle_timeout=300)
print("Hibernating model demo class initialized.")